# Week 22: Multimodal Foundation Models & Vision-Language Architectures
## Notebook 05: Video Diffusion Models & Spatio-Temporal Attention from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Generating video (Sora, Runway Gen-2, Stable Video Diffusion) is vastly more complex than generating static images:
- Generating 24 independent image frames causes catastrophic temporal flickering and jitter.
- Video requires **Temporal Consistency**: objects must maintain identity, physics, and smooth motion trajectories across frames.

Modern Video Diffusion Models extend 2D Diffusion architectures by adding:
1. **Spatial Attention:** Operates within each individual frame (capturing visual texture and scene layout).
2. **Temporal Attention:** Operates across time frames for each spatial coordinate (tracking pixel movement and motion physics across time)!

---

### 2. Mathematical Foundations: Factorized Spatio-Temporal Attention
For a video latent tensor $X \in \mathbb{R}^{B \times T \times H \times W \times C}$:

1. **Spatial Attention (Reshape to $(B \cdot T, H \cdot W, C)$):**
   $$X_{\text{spatial}} = \text{Attention}(X, X, X)$$
   Applies standard 2D attention within each frame independently.

2. **Temporal Attention (Reshape to $(B \cdot H \cdot W, T, C)$):**
   $$X_{\text{temporal}} = \text{Attention}(X, X, X)$$
   Every pixel coordinate $(h, w)$ attends to the same coordinate across all $T$ time frames!
   Complexity scales with $O(T^2)$ instead of an intractable monolithic $O((T \cdot H \cdot W)^2)$!

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Video Latent Representation
# Batch B=1, Frames T=3, Height H=2, Width W=2, Channels C=4
B, T, H, W, C = 1, 3, 2, 2, 4
video_latents = np.random.randn(B, T, H, W, C)

print(f"Video Latent Shape: [Batch={B}, Frames={T}, Height={H}, Width={W}, Channels={C}]")

# Step 2: Implement Temporal Self-Attention Layer
def temporal_self_attention(x):
    B, T, H, W, C = x.shape
    # Reshape: group spatial coordinates together -> (B * H * W, T, C)
    # Permute to (B, H, W, T, C) then flatten spatial dimensions
    x_permuted = np.transpose(x, (0, 2, 3, 1, 4)) # (B, H, W, T, C)
    x_temporal = x_permuted.reshape(B * H * W, T, C)
    
    # Query, Key, Value Projections across Time
    W_q = np.random.randn(C, C) * 0.2
    W_k = np.random.randn(C, C) * 0.2
    W_v = np.random.randn(C, C) * 0.2
    
    Q = x_temporal @ W_q
    K = x_temporal @ W_k
    V = x_temporal @ W_v
    
    # Attention across T frames: (B*H*W, T, T)
    scores = (Q @ np.transpose(K, (0, 2, 1))) / np.sqrt(C)
    # Softmax over time dimension
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attn_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Apply attention to values
    out_temporal = attn_weights @ V # (B*H*W, T, C)
    
    # Invert reshape back to original (B, T, H, W, C)
    out_reshaped = out_temporal.reshape(B, H, W, T, C)
    out_final = np.transpose(out_reshaped, (0, 3, 1, 2, 4))
    
    return out_final, attn_weights

out_video, temporal_attn = temporal_self_attention(video_latents)

print(f"Output Video Shape: {out_video.shape} (Preserved video dimensions!)")
print(f"\nTemporal Attention Matrix for Pixel (y=0, x=0) across 3 frames:\n", np.round(temporal_attn[0], 3))
print("Row sum:", np.sum(temporal_attn[0], axis=-1))

### 3. Step-by-Step Code Walkthrough
1. **Dimension Permutation:** Transposing to $(B \cdot H \cdot W, T, C)$ decouples temporal modeling from spatial modeling.
2. **Cross-Frame Alignment:** Every individual pixel location tracks its trajectory across all $T$ video frames.
3. **Preserved Tensor Geometry:** The final output preserves exact video tensor dimensions $(B, T, H, W, C)$.

---

### 4. Industry Applications & Trade-offs
- **OpenAI Sora & Stable Video Diffusion:** Factorized spatial-temporal transformer blocks generate photorealistic physics simulation.
- **Trade-off:** VRAM scaling: generating 5-second 1080p video at 24 FPS requires 120 frames, requiring aggressive 3D VAE latent compression.

---

### 5. Top Interview Questions & Answers
**Q1: Why is factorized spatio-temporal attention used instead of full 3D attention?**
> **Answer:** Full 3D attention computes pairwise attention across all space and time tokens simultaneously. For a video with $T=32$ and $H \times W = 64 \times 64$, total tokens $N = 131,072$. Full attention requires $(131,072)^2 \approx 17$ billion matrix cells (crashing any GPU). Factorized attention computes spatial attention $O((HW)^2)$ followed by temporal attention $O(T^2)$, reducing compute by over 99%.

**Q2: What is 3D VAE Latent Causal Packing in video models?**
> **Answer:** A 3D VAE compresses video both spatially ($8 \times 8$ pixel compression) and temporally (compressing every 4 time frames into 1 latent frame). Causal convolutions ensure that temporal compression does not leak information from future frames into past frames.